In [10]:
# ============================================
# FEATURE ENGINEERING - WEEK 3-4 (FIXED)
# Criminal Appeal Outcome Prediction
# ============================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("FEATURE ENGINEERING - WEEK 3-4")
print("=" * 70)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ============================================
# STEP 1: LOAD CLEANED DATASET
# ============================================
print("STEP 1: Loading cleaned dataset...")
print("-" * 70)

df = pd.read_csv('dataset_cleaned_v2.csv')
print(f"✅ Loaded: {len(df)} records, {len(df.columns)} features")
print(f"   Columns available: {len(df.columns)}")
print()

FEATURE ENGINEERING - WEEK 3-4
Started: 2026-01-06 14:39:23

STEP 1: Loading cleaned dataset...
----------------------------------------------------------------------
✅ Loaded: 1251 records, 98 features
   Columns available: 98



In [11]:
# ============================================
# STEP 2: TEXT FEATURE EXTRACTION
# ============================================
print("STEP 2: Extracting text features...")
print("-" * 70)

# Identify text columns that exist
text_columns = []
for col in ['brief_facts_summary', 'grounds_of_appeal_raw_text_summary', 'court_of_appeal_analysis_summary']:
    if col in df.columns:
        text_columns.append(col)
        df[col] = df[col].fillna('')

print(f"Text columns found: {len(text_columns)}")
print(f"   {text_columns}")
print()


STEP 2: Extracting text features...
----------------------------------------------------------------------
Text columns found: 3
   ['brief_facts_summary', 'grounds_of_appeal_raw_text_summary', 'court_of_appeal_analysis_summary']



In [13]:
# ============================================
# 2A: TF-IDF VECTORIZATION
# ============================================
print("2A: Creating TF-IDF features...")

# Combine all text into single document per case
text_parts = []
for col in text_columns:
    text_parts.append(df[col].fillna(''))

df['combined_text'] = ' '.join([' '.join(parts) for parts in zip(*text_parts)])

# Configure TF-IDF parameters
tfidf_params = {
    'max_features': 50,   # Reduced to 50 most important terms
    'min_df': 2,          # Term must appear in at least 3 documents
    'max_df': 0.95,        # Ignore terms appearing in >80% of documents
    'ngram_range': (1, 2), # Unigrams and bigrams
    'stop_words': 'english',
    'sublinear_tf': True   # Apply log scaling to term frequency
}

# Fit TF-IDF vectorizer
tfidf = TfidfVectorizer(**tfidf_params)
tfidf_matrix = tfidf.fit_transform(df['combined_text'])

# Convert to DataFrame
tfidf_features = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
)

print(f"✅ Created {tfidf_matrix.shape[1]} TF-IDF features")
print()


2A: Creating TF-IDF features...


ValueError: After pruning, no terms remain. Try a lower min_df or a higher max_df.

In [14]:
# ============================================
# 2A: TF-IDF VECTORIZATION (SKIP VERSION)
# ============================================
print("2A: Skipping TF-IDF (using text statistics only)...")

# Create empty TF-IDF features
tfidf_features = pd.DataFrame()
print("⚠️ TF-IDF skipped - will use other features for modeling")
print()


2A: Skipping TF-IDF (using text statistics only)...
⚠️ TF-IDF skipped - will use other features for modeling



In [15]:
# ============================================
# 2B: TEXT STATISTICS
# ============================================
print("2B: Creating text statistics...")

text_stat_features = []

for col in text_columns:
    length_col = f'{col}_length'
    word_col = f'{col}_word_count'
    
    df[length_col] = df[col].str.len().fillna(0)
    df[word_col] = df[col].str.split().str.len().fillna(0)
    
    text_stat_features.extend([length_col, word_col])

print(f"✅ Created {len(text_stat_features)} text statistics")
print()

# ============================================
# STEP 3: CATEGORICAL ENCODING
# ============================================
print("STEP 3: Encoding categorical variables...")
print("-" * 70)

# ============================================
# 3A: ONE-HOT ENCODING
# ============================================
print("3A: One-hot encoding...")

# Check which categorical columns exist
categorical_cols = []
for col in ['offence_category_grouped', 'medical_evidence_strength', 
            'chain_of_custody_quality', 'appeal_type_simplified']:
    if col in df.columns:
        categorical_cols.append(col)

print(f"Categorical columns found: {categorical_cols}")

# Create dummy variables
encoded_features = pd.get_dummies(
    df[categorical_cols],
    prefix=categorical_cols,
    drop_first=True  # Avoid multicollinearity
)

print(f"✅ Created {len(encoded_features.columns)} one-hot encoded features")
print()


2B: Creating text statistics...
✅ Created 6 text statistics

STEP 3: Encoding categorical variables...
----------------------------------------------------------------------
3A: One-hot encoding...
Categorical columns found: ['offence_category_grouped', 'medical_evidence_strength', 'chain_of_custody_quality', 'appeal_type_simplified']
✅ Created 21 one-hot encoded features



In [16]:
# ============================================
# 3B: LABEL ENCODING FOR TARGET
# ============================================
print("3B: Encoding target variables...")

# Encode outcome_clean
le_outcome = LabelEncoder()
df['outcome_encoded'] = le_outcome.fit_transform(df['outcome_clean'])

# Encode conviction_clean
le_conviction = LabelEncoder()
df['conviction_encoded'] = le_conviction.fit_transform(df['conviction_clean'])

# Encode combined_outcome
le_combined = LabelEncoder()
df['combined_outcome_encoded'] = le_combined.fit_transform(df['combined_outcome'])

print("✅ Encoded target variables:")
print(f"   outcome_clean: {dict(enumerate(le_outcome.classes_))}")
print(f"   conviction_clean: {dict(enumerate(le_conviction.classes_))}")
print()


3B: Encoding target variables...
✅ Encoded target variables:
   outcome_clean: {0: 'Appeal_Allowed', 1: 'Appeal_Dismissed', 2: 'Partly_Allowed'}
   conviction_clean: {0: 'Acquitted', 1: 'Convicted', 2: 'Modified', 3: 'Remanded'}



In [17]:
# ============================================
# STEP 4: INTERACTION FEATURES
# ============================================
print("STEP 4: Creating interaction features...")
print("-" * 70)

interaction_features = []

# Create offense-evidence interactions (only if columns exist)
if 'eyewitness_present' in df.columns and 'offence_category_grouped' in df.columns:
    df['murder_with_eyewitness'] = (
        (df['offence_category_grouped'] == 'Murder_Related') & 
        (df['eyewitness_present'] == 'Yes')
    ).astype(int)
    interaction_features.append('murder_with_eyewitness')

if 'forensic_evidence_present' in df.columns and 'offence_category_grouped' in df.columns:
    df['drug_with_forensic'] = (
        (df['offence_category_grouped'] == 'Drug_Related') & 
        (df['forensic_evidence_present'] == 'Yes')
    ).astype(int)
    interaction_features.append('drug_with_forensic')

# Evidence strength composite score
evidence_strength_map = {'Strong': 3, 'Moderate': 2, 'Weak': 1, 'Unknown': 0}
if 'medical_evidence_strength' in df.columns:
    df['medical_evidence_score'] = df['medical_evidence_strength'].map(evidence_strength_map).fillna(0)
    interaction_features.append('medical_evidence_score')

custody_quality_map = {'Good': 3, 'Moderate': 2, 'Weak': 1, 'Unknown': 0}
if 'chain_of_custody_quality' in df.columns:
    df['custody_quality_score'] = df['chain_of_custody_quality'].map(custody_quality_map).fillna(0)
    interaction_features.append('custody_quality_score')

# Evidence count
evidence_cols = [col for col in df.columns if 'evidence' in col.lower() and col.endswith('_present')]
if evidence_cols:
    df['evidence_count'] = df[evidence_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)
    interaction_features.append('evidence_count')

# Grounds-evidence interaction
if 'total_grounds_raised' in df.columns and 'evidence_count' in interaction_features:
    df['grounds_to_evidence_ratio'] = df['total_grounds_raised'] / (df['evidence_count'] + 1)
    interaction_features.append('grounds_to_evidence_ratio')

print(f"✅ Created {len(interaction_features)} interaction features")
print(f"   Features: {interaction_features}")
print()


STEP 4: Creating interaction features...
----------------------------------------------------------------------
✅ Created 6 interaction features
   Features: ['murder_with_eyewitness', 'drug_with_forensic', 'medical_evidence_score', 'custody_quality_score', 'evidence_count', 'grounds_to_evidence_ratio']



In [18]:
# ============================================
# STEP 5: SELECT FEATURES FOR MODELING
# ============================================
print("STEP 5: Selecting features for modeling...")
print("-" * 70)

# Select ground columns
ground_cols = [col for col in df.columns if col.startswith('gnd_') and col != 'gnd_other_description']

# Select evidence columns  
evidence_present_cols = [col for col in df.columns if col.endswith('_present')]

# Select numerical features that exist
numerical_cols = []
for col in ['total_grounds_raised', 'coa_year', 'appeal_duration_days']:
    if col in df.columns:
        numerical_cols.append(col)

# Add text statistics and interactions
numerical_cols.extend(text_stat_features)
numerical_cols.extend(interaction_features)

print(f"Feature breakdown:")
print(f"   - Ground features: {len(ground_cols)}")
print(f"   - Evidence features: {len(evidence_present_cols)}")
print(f"   - Numerical features: {len(numerical_cols)}")
print(f"   - Categorical encoded: {len(encoded_features.columns)}")
print()

# Combine all feature lists (only existing features)
feature_columns = ground_cols + evidence_present_cols + numerical_cols

# Verify all features exist
missing_features = [col for col in feature_columns if col not in df.columns]
if missing_features:
    print(f"⚠️ Removing {len(missing_features)} missing features...")
    feature_columns = [col for col in feature_columns if col in df.columns]

# Create feature matrix
X = df[feature_columns].copy()

# Handle Yes/No columns
yes_no_cols = [col for col in X.columns if X[col].dtype == 'object']
for col in yes_no_cols:
    X[col] = (X[col] == 'Yes').astype(int)

# Fill remaining missing values
X = X.fillna(0)

print(f"✅ Selected {len(feature_columns)} base features")
print()


STEP 5: Selecting features for modeling...
----------------------------------------------------------------------
Feature breakdown:
   - Ground features: 14
   - Evidence features: 9
   - Numerical features: 15
   - Categorical encoded: 21

✅ Selected 38 base features



In [19]:
# ============================================
# STEP 6: ADD ENCODED & TF-IDF FEATURES
# ============================================
print("STEP 6: Combining all features...")
print("-" * 70)

# Reset indices to ensure alignment
X = X.reset_index(drop=True)
encoded_features = encoded_features.reset_index(drop=True)
tfidf_features = tfidf_features.reset_index(drop=True)

# Combine all features
X_combined = pd.concat([X, encoded_features, tfidf_features], axis=1)

print(f"✅ Total features: {len(X_combined.columns)}")
print(f"   - Base features: {len(X.columns)}")
print(f"   - Encoded features: {len(encoded_features.columns)}")
print(f"   - TF-IDF features: {len(tfidf_features.columns)}")
print()


STEP 6: Combining all features...
----------------------------------------------------------------------
✅ Total features: 59
   - Base features: 38
   - Encoded features: 21
   - TF-IDF features: 0



In [20]:
# ============================================
# STEP 7: FEATURE SCALING
# ============================================
print("STEP 7: Scaling numerical features...")
print("-" * 70)

# Identify columns to scale (numerical features + TF-IDF)
cols_to_scale = numerical_cols + list(tfidf_features.columns)

# Remove any columns not in X_combined
cols_to_scale = [col for col in cols_to_scale if col in X_combined.columns]

# Create scaler
scaler = StandardScaler()

# Scale features
X_combined[cols_to_scale] = scaler.fit_transform(X_combined[cols_to_scale])

print(f"✅ Scaled {len(cols_to_scale)} numerical features")
print()



STEP 7: Scaling numerical features...
----------------------------------------------------------------------
✅ Scaled 15 numerical features



In [21]:
# ============================================
# STEP 8: TRAIN/TEST SPLIT
# ============================================
print("STEP 8: Creating train/test split...")
print("-" * 70)

# Prepare target variables
y_outcome = df['outcome_encoded'].values
y_conviction = df['conviction_encoded'].values
y_combined = df['combined_outcome_encoded'].values

# Temporal validation split
# Sort by COA year (earlier cases = training, later = testing)
if 'coa_year' in df.columns:
    df_sorted = df.sort_values('coa_year').reset_index(drop=True)
    sort_indices = df_sorted.index
else:
    # Fallback: random split if no year column
    df_sorted = df.reset_index(drop=True)
    sort_indices = df_sorted.index

X_combined_sorted = X_combined.loc[sort_indices].reset_index(drop=True)

# 80/20 split
split_idx = int(len(df_sorted) * 0.8)

X_train = X_combined_sorted.iloc[:split_idx]
X_test = X_combined_sorted.iloc[split_idx:]

y_train_outcome = df_sorted['outcome_encoded'].iloc[:split_idx].values
y_test_outcome = df_sorted['outcome_encoded'].iloc[split_idx:].values

y_train_conviction = df_sorted['conviction_encoded'].iloc[:split_idx].values
y_test_conviction = df_sorted['conviction_encoded'].iloc[split_idx:].values

y_train_combined = df_sorted['combined_outcome_encoded'].iloc[:split_idx].values
y_test_combined = df_sorted['combined_outcome_encoded'].iloc[split_idx:].values

print("✅ Train/Test split (temporal validation):")
print(f"   Training set: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"   Test set: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")
print()

# Check class distribution
print("Target distribution (outcome_clean):")
print("Training set:")
train_dist = pd.Series(y_train_outcome).value_counts().sort_index()
for idx, count in train_dist.items():
    print(f"   {le_outcome.classes_[idx]}: {count} ({count/len(y_train_outcome)*100:.1f}%)")

print("\nTest set:")
test_dist = pd.Series(y_test_outcome).value_counts().sort_index()
for idx, count in test_dist.items():
    print(f"   {le_outcome.classes_[idx]}: {count} ({count/len(y_test_outcome)*100:.1f}%)")
print()



STEP 8: Creating train/test split...
----------------------------------------------------------------------
✅ Train/Test split (temporal validation):
   Training set: 1000 samples (79.9%)
   Test set: 251 samples (20.1%)

Target distribution (outcome_clean):
Training set:
   Appeal_Allowed: 392 (39.2%)
   Appeal_Dismissed: 485 (48.5%)
   Partly_Allowed: 123 (12.3%)

Test set:
   Appeal_Allowed: 113 (45.0%)
   Appeal_Dismissed: 103 (41.0%)
   Partly_Allowed: 35 (13.9%)



In [22]:
# ============================================
# STEP 9: SAVE DATASETS & ARTIFACTS
# ============================================
print("STEP 9: Saving datasets and artifacts...")
print("-" * 70)

# Save train/test sets
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)

np.save('y_train_outcome.npy', y_train_outcome)
np.save('y_test_outcome.npy', y_test_outcome)
np.save('y_train_conviction.npy', y_train_conviction)
np.save('y_test_conviction.npy', y_test_conviction)
np.save('y_train_combined.npy', y_train_combined)
np.save('y_test_combined.npy', y_test_combined)

print("✅ Saved training and test datasets:")
print("   - X_train.csv, X_test.csv")
print("   - y_train_outcome.npy, y_test_outcome.npy")
print("   - y_train_conviction.npy, y_test_conviction.npy")
print("   - y_train_combined.npy, y_test_combined.npy")
print()

# Save preprocessing artifacts
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('label_encoder_outcome.pkl', 'wb') as f:
    pickle.dump(le_outcome, f)

with open('label_encoder_conviction.pkl', 'wb') as f:
    pickle.dump(le_conviction, f)

with open('label_encoder_combined.pkl', 'wb') as f:
    pickle.dump(le_combined, f)

print("✅ Saved preprocessing artifacts:")
print("   - tfidf_vectorizer.pkl")
print("   - scaler.pkl")
print("   - label_encoder_*.pkl")
print()

STEP 9: Saving datasets and artifacts...
----------------------------------------------------------------------
✅ Saved training and test datasets:
   - X_train.csv, X_test.csv
   - y_train_outcome.npy, y_test_outcome.npy
   - y_train_conviction.npy, y_test_conviction.npy
   - y_train_combined.npy, y_test_combined.npy

✅ Saved preprocessing artifacts:
   - tfidf_vectorizer.pkl
   - scaler.pkl
   - label_encoder_*.pkl



In [23]:
# ============================================
# STEP 10: CREATE FEATURE DOCUMENTATION
# ============================================
print("STEP 10: Creating feature documentation...")
print("-" * 70)

feature_metadata = {
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_samples': int(len(df)),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'total_features': int(len(X_combined.columns)),
    'feature_categories': {
        'grounds_of_appeal': int(len(ground_cols)),
        'evidence_present': int(len(evidence_present_cols)),
        'numerical_base': int(len([c for c in numerical_cols if c in X_combined.columns])),
        'categorical_encoded': int(len(encoded_features.columns)),
        'tfidf_text': int(tfidf_matrix.shape[1]),
        'interaction': int(len(interaction_features))
    },
    'feature_names': list(X_combined.columns),
    'target_variables': {
        'outcome_clean': list(le_outcome.classes_),
        'conviction_clean': list(le_conviction.classes_),
        'combined_outcome': list(le_combined.classes_)
    },
    'tfidf_params': tfidf_params,
    'split_method': 'temporal_validation',
    'split_ratio': '80/20'
}

with open('feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=4)

print("✅ Saved feature_metadata.json")
print()

STEP 10: Creating feature documentation...
----------------------------------------------------------------------
✅ Saved feature_metadata.json



In [24]:
# ============================================
# SUMMARY
# ============================================
print("=" * 70)
print("🎉 FEATURE ENGINEERING COMPLETE!")
print("=" * 70)
print()
print("Summary:")
print(f"  Total features created: {len(X_combined.columns)}")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print()
print("Feature breakdown:")
for category, count in feature_metadata['feature_categories'].items():
    print(f"  - {category}: {count}")
print()
print("Outputs created:")
print("  ✅ X_train.csv, X_test.csv (feature matrices)")
print("  ✅ y_train_*.npy, y_test_*.npy (target variables)")
print("  ✅ Preprocessing artifacts (TF-IDF, scaler, encoders)")
print("  ✅ feature_metadata.json (documentation)")
print()
print("=" * 70)
print("✅ READY FOR MODELING (WEEK 5-6)!")
print("=" * 70)
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

🎉 FEATURE ENGINEERING COMPLETE!

Summary:
  Total features created: 59
  Training samples: 1000
  Test samples: 251

Feature breakdown:
  - grounds_of_appeal: 14
  - evidence_present: 9
  - numerical_base: 15
  - categorical_encoded: 21
  - tfidf_text: 100
  - interaction: 6

Outputs created:
  ✅ X_train.csv, X_test.csv (feature matrices)
  ✅ y_train_*.npy, y_test_*.npy (target variables)
  ✅ Preprocessing artifacts (TF-IDF, scaler, encoders)
  ✅ feature_metadata.json (documentation)

✅ READY FOR MODELING (WEEK 5-6)!
Completed: 2026-01-06 16:02:23
